# UML Diagram Analyzer — Fine-tuning com Unsloth

Fine-tunes **Qwen2-VL-2B** no dataset de diagramas UML gerado localmente, usando o Unsloth para treinamento LoRA rápido.  
Ao final, o modelo é exportado em GGUF para uso local com **Ollama**.

**Pré-requisitos:**
- Google Colab com GPU (T4 mínimo, A100 recomendado)
- `data/finetune_chat.jsonl` carregado na sessão do Colab

## 1. Instalação das dependências

Antes de começar, precisamos instalar as bibliotecas necessárias. O **Unsloth** é a principal ferramenta aqui — ele permite fazer fine-tuning de modelos grandes de forma muito mais rápida e com menos memória do que as abordagens tradicionais, graças a otimizações específicas para GPUs. As demais bibliotecas (`trl`, `peft`, `accelerate`, `bitsandbytes`) são dependências que viabilizam técnicas como LoRA e quantização de 4 bits.

O `%%capture` no início suprime a saída da instalação, que costuma ser longa. Dependendo do ambiente do Colab, essa etapa pode levar alguns minutos na primeira execução.

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets pillow

## 2. Imports e configuração

Importamos as ferramentas e definimos as configurações principais. O modelo escolhido é o `Qwen2-VL-2B-Instruct`, um modelo de visão e linguagem com 2 bilhões de parâmetros — pequeno o suficiente para caber no Colab gratuito, mas ainda capaz de processar imagens e gerar texto. A versão `bnb-4bit` já vem quantizada, o que reduz o uso de memória sem precisar fazer nada extra.

O `MAX_SEQ_LENGTH` define o tamanho máximo das sequências de tokens que o modelo processa de uma vez. Valores maiores permitem mais contexto, mas consomem mais VRAM.

In [ ]:
import torch
import json
import base64
import io
from pathlib import Path
from PIL import Image
from datasets import Dataset
from unsloth import FastVisionModel
from trl import SFTTrainer, SFTConfig
from unsloth.trainer import UnslothVisionDataCollator

MODEL_NAME = "unsloth/Qwen2-VL-2B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DATASET_PATH = "data/finetune_chat.jsonl"
OUTPUT_DIR = "model_output"

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"BF16 supported: {torch.cuda.is_bf16_supported()}")

## 3. Carregamento do modelo e configuração do LoRA

Carregamos o modelo pré-treinado e configuramos os adaptadores **LoRA** (Low-Rank Adaptation). O LoRA é uma técnica que evita re-treinar o modelo inteiro — em vez disso, ele adiciona pequenas matrizes treináveis em camadas específicas, tornando o processo muito mais leve. O parâmetro `r=16` controla o tamanho dessas matrizes; valores maiores capturam mais nuances, mas exigem mais memória.

Ativamos o fine-tuning em todas as camadas — visão, linguagem, atenção e MLP — para que o modelo aprenda a correlacionar melhor o que vê nas imagens com as respostas textuais esperadas sobre diagramas UML.

In [ ]:
model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    random_state=42,
)

print("Modelo carregado e LoRA configurado.")

## 4. Carregamento e conversão do dataset

Nosso dataset está no formato JSONL com imagens codificadas em base64. Lemos cada linha, decodificamos a imagem para o formato PIL e reorganizamos os dados no formato de conversa que o Unsloth espera — pares de mensagem do usuário (imagem + instrução de texto) e resposta do assistente.

Esse formato imita um chat porque o modelo foi pré-treinado nesse padrão. Converter nossos dados para a mesma estrutura ajuda o modelo a entender o que esperamos dele — é como "falar a língua" que ele já conhece antes de ensinar algo novo.

In [ ]:
def load_dataset_from_jsonl(path: str) -> Dataset:
    """Lê finetune_chat.jsonl e converte para o formato de conversa do Unsloth."""
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            user_content = rec["messages"][0]["content"]
            assistant_content = rec["messages"][1]["content"]

            # Decodifica imagem base64 → PIL Image
            b64_url = user_content[0]["image_url"]["url"]
            b64_data = b64_url.split(",")[1]
            img = Image.open(io.BytesIO(base64.b64decode(b64_data))).convert("RGB")

            text_instruction = user_content[1]["text"]

            records.append({
                "messages": [
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": img},
                            {"type": "text", "text": text_instruction},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [{"type": "text", "text": assistant_content}],
                    },
                ]
            })

    return Dataset.from_list(records)


dataset = load_dataset_from_jsonl(DATASET_PATH)
print(f"Exemplos carregados: {len(dataset)}")
print(f"Chaves do exemplo: {list(dataset[0].keys())}")

## 5. Treinamento

Configuramos e executamos o treinamento com o `SFTTrainer` (Supervised Fine-Tuning Trainer). O `gradient_accumulation_steps=4` junta os gradientes de 4 batches antes de atualizar os pesos — isso simula um batch maior sem precisar de mais memória. Usamos 60 passos, o suficiente para o modelo começar a se especializar em UML sem demorar horas no Colab gratuito.

O `learning_rate=2e-4` com `warmup_steps=5` faz o modelo aprender de forma gradual, começando devagar e acelerando conforme progride. O otimizador `adamw_8bit` é uma versão compacta do Adam clássico, projetada para ocupar menos memória em GPUs.

In [ ]:
FastVisionModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    args=SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=42,
        output_dir=OUTPUT_DIR,
        report_to="none",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True},
    ),
)

trainer_stats = trainer.train()
print(f"Treinamento concluído. Passos: {trainer_stats.global_step}")

## 6. Salvando os adaptadores LoRA

Salvamos apenas os **adaptadores LoRA**, não o modelo completo. Isso é bem mais eficiente: enquanto o modelo base tem bilhões de parâmetros, os adaptadores são apenas uma fração disso — geralmente alguns megabytes. Eles funcionam como um "patch" que é aplicado em cima do modelo base na hora de usar.

Essa abordagem também facilita o compartilhamento: em vez de distribuir gigabytes de modelo, você compartilha só os adaptadores e referencia o modelo base público.

In [ ]:
model.save_pretrained("uml_analyzer_lora")
tokenizer.save_pretrained("uml_analyzer_lora")
print("Adaptadores LoRA salvos em uml_analyzer_lora/")

## 7. Exportando para GGUF (Ollama)

Para usar o modelo localmente com o Ollama, precisamos convertê-lo para o formato **GGUF** (GPT-Generated Unified Format), que é otimizado para rodar em hardware consumer — inclusive só na CPU, sem GPU. O método `save_pretrained_gguf` do Unsloth cuida de mesclar o modelo base com os adaptadores LoRA e converter tudo automaticamente.

Usamos a quantização `q4_k_m`, que comprime o modelo para 4 bits com um algoritmo de boa qualidade. O resultado é um arquivo bem menor (de ~4 GB para ~1.5 GB) com perda mínima de qualidade — um equilíbrio razoável para uso local.

In [ ]:
model.save_pretrained_gguf(
    "uml_analyzer_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("Modelo GGUF salvo em uml_analyzer_gguf/ — pronto para o Ollama")

## 8. Gerando o Modelfile para o Ollama

O **Modelfile** é o arquivo de configuração que o Ollama usa para entender como carregar e executar o modelo. Ele define qual arquivo GGUF usar, o system prompt (instrução que fica ativa em todas as conversas) e parâmetros de geração como `temperature`. Um `temperature=0.1` deixa as respostas mais determinísticas e precisas — o que faz sentido para análise técnica de diagramas.

Depois de baixar o `uml_analyzer_gguf/` e o `Modelfile` do Colab para a sua máquina, basta rodar `ollama create uml-analyzer -f Modelfile` e o modelo estará disponível localmente via `ollama run uml-analyzer`.

In [ ]:
modelfile_content = """FROM ./uml_analyzer_gguf/uml_analyzer-Q4_K_M.gguf

SYSTEM You are an expert UML diagram analyzer. Given an image of a UML diagram, you provide accurate and detailed analysis of its structure, entities, relationships, and purpose.

PARAMETER temperature 0.1
PARAMETER top_p 0.9
PARAMETER stop "<|endoftext|>"
"""

Path("Modelfile").write_text(modelfile_content, encoding="utf-8")

print("Modelfile criado.")
print()
print("Próximos passos (executar localmente após baixar os arquivos do Colab):")
print("  ollama create uml-analyzer -f Modelfile")
print("  ollama run uml-analyzer")